# Example: Diagnosing Stylized Facts in Daily Equity Growth Rates
This example turns the three empirical facts from L3a into a reproducible model-checking workflow: heavy tails, weak linear autocorrelation in growth rates, and persistent autocorrelation in growth-rate magnitude.

> __Learning Objectives:__
>
> * __Compare distributional models:__ Rank Normal, Laplace, and Student-t descriptions without mistaking a large-sample goodness-of-fit test for a binary model certificate.
> * __Estimate tail thickness responsibly:__ Compute Hill estimates over a range of order-statistic cutoffs and inspect stability instead of reporting one arbitrary tail index.
> * __Separate return memory from volatility memory:__ Compare the ACF of $g_t$ with the ACF of $|g_t|$.
> * __Translate diagnostics into model choices:__ State what a Gaussian SIM can represent and what a credible stress generator must add.

___


## Setup, Data, and Prerequisites
We use the frozen course market dataset and the reusable diagnostics in VLQuantitativeFinancePackage.jl. The course convention is $g_t=\Delta t^{-1}\log(P_t/P_{t-1})$; the corresponding one-day log return is $r_t=\Delta t\,g_t$.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


In [ ]:
raw_dataset = MyTrainingMarketDataSet()["dataset"];
maximum_days = nrow(raw_dataset["AAPL"]);
dataset = Dict(ticker => frame for (ticker, frame) in raw_dataset if nrow(frame) == maximum_days);
tickers = sort(collect(keys(dataset)));
Δt = 1 / 252;
growth_rates = log_growth_matrix(dataset, tickers; Δt=Δt, risk_free_rate=0.0);

ticker = "AMD";
ticker_index = findfirst(==(ticker), tickers);
g = growth_rates[:, ticker_index];
r = Δt .* g;
println("$(ticker): $(length(g)) daily observations from a $(length(tickers))-ticker complete-case universe")


## Task 1: Inspect the Series Before Fitting a Model
A distribution plot alone discards time order. We begin with the growth-rate path and a compact diagnostic report at economically interpretable lags.


In [ ]:
diagnostic_lags = [1, 5, 10, 20, 50];
report = stylized_facts_report(g; lags=diagnostic_lags);
diagnostics = DataFrame(
    lag=diagnostic_lags,
    raw_growth_acf=report.raw_acf,
    absolute_growth_acf=report.absolute_acf,
    squared_growth_acf=report.squared_acf,
);
pretty_table(diagnostics; table_format=TextTableFormat(borders=text_table_borders__simple));
println("Absolute-tail Hill estimate: $(round(report.tail_index, digits=2))")

plot(g; label=ticker, c=:navy, lw=1, xlabel="Trading-day index",
    ylabel="Annualized growth-rate observation (1/yr)",
    title="Daily growth-rate observations", framestyle=:box)


## Task 2: Compare Thin- and Heavy-Tailed Distributions
We fit Normal and Laplace models by maximum likelihood and profile a standardized Student-t model over a small degrees-of-freedom grid. The Anderson--Darling statistic is used as a relative discrepancy score. Because parameters are estimated from the same sample—and because thousands of observations make every test powerful—the nominal p-values are not treated as pass/fail certificates.


In [ ]:
normal_fit = fit_mle(Normal, r);
laplace_fit = fit_mle(Laplace, r);
ν_grid = [2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 15.0, 30.0];
z = (r .- mean(r)) ./ std(r);
ν = ν_grid[argmax([sum(logpdf.(TDist(candidate), z)) for candidate in ν_grid])];
student_fit = LocationScale(mean(r), std(r) * sqrt((ν - 2) / ν), TDist(ν));

fits = [("Normal", normal_fit), ("Laplace", laplace_fit), ("Student-t (ν=$(ν))", student_fit)];
fit_table = DataFrame(
    distribution=first.(fits),
    AD_statistic=[OneSampleADTest(r, distribution).A² for (_, distribution) in fits],
    log_likelihood=[sum(logpdf.(distribution, r)) for (_, distribution) in fits],
);
sort!(fit_table, :AD_statistic);
pretty_table(fit_table; table_format=TextTableFormat(borders=text_table_borders__simple))

p = plot(normal_fit; label="Normal", c=:deepskyblue, lw=3,
    xlabel="Daily log return r", ylabel="Density", framestyle=:box);
plot!(p, laplace_fit; label="Laplace", c=:gray35, lw=3);
plot!(p, student_fit; label="Student-t", c=:green4, lw=3, ls=:dash);
density!(p, r; label="Observed", c=:red, lw=3)


### Hill Stability Plot
Hill estimation has a bias--variance tradeoff: too few extremes produce a noisy estimate, while too many admit observations from the body of the distribution. A stability plot makes that choice visible. We apply the estimator to $|g_t|$, so the diagnostic summarizes both tails.


In [ ]:
number_of_tail_observations = count(>(0), abs.(g));
maximum_k = min(300, number_of_tail_observations - 1);
hill_k = unique(round.(Int, range(20, maximum_k; length=30)));
hill_alpha = [hill_tail_index(g; k=k, tail=:absolute) for k in hill_k];
default_k = floor(Int, sqrt(number_of_tail_observations));
default_alpha = hill_tail_index(g; k=default_k, tail=:absolute);

plot(hill_k, hill_alpha; marker=:circle, ms=3, c=:purple, label="Hill estimate",
    xlabel="Number k of upper order statistics", ylabel="Tail index α",
    title="Hill stability plot for |g|", framestyle=:box);
vline!([default_k]; c=:black, ls=:dash, label="k=floor(sqrt(n))");
hline!([default_alpha]; c=:red, ls=:dot, label="α=$(round(default_alpha, digits=2))")


## Task 3: Separate Directional Memory from Volatility Memory
The random-walk null concerns linear dependence in $g_t$, not independence of the entire process. Near-zero raw-growth ACF can coexist with slowly decaying ACF in $|g_t|$; that combination is the signature of volatility clustering.


In [ ]:
lags = collect(0:100);
raw_acf = sample_autocorrelation(g, lags);
absolute_acf = sample_autocorrelation(abs.(g), lags);
confidence = 2.576 / sqrt(length(g));

p1 = plot(lags, raw_acf; c=:navy, lw=2, label="ACF(g)",
    xlabel="Lag (trading days)", ylabel="Autocorrelation", title="Directional memory", framestyle=:box);
hline!(p1, [confidence, -confidence]; c=:black, ls=:dash, label="99% white-noise band");
p2 = plot(lags, absolute_acf; c=:red, lw=2, label="ACF(|g|)",
    xlabel="Lag (trading days)", ylabel="Autocorrelation", title="Volatility memory", framestyle=:box);
hline!(p2, [confidence, -confidence]; c=:black, ls=:dash, label="99% white-noise band");
plot(p1, p2; layout=(1, 2), size=(1000, 400))


## Task 4: Model-Adequacy Handoff
These diagnostics assign different jobs to different models.

- A Gaussian SIM is a useful low-parameter model of contemporaneous cross-asset covariance and market exposure.
- Heavy tails warn against using Normal bands as complete tail-risk measures; empirical quantiles, VaR/CVaR, or heavy-tailed innovations are needed for stress work.
- Volatility clustering warns that independently resampling single-day residuals will not reproduce multi-day risk persistence; block resampling or a time-varying/regime model is needed when path shape matters.

This distinction carries directly into L6 SIM uncertainty quantification: empirical residual bootstraps preserve the one-day residual distribution, but neither an i.i.d. residual bootstrap nor a Gaussian parametric bootstrap preserves volatility regimes.


## Summary

> __Key Takeaways:__
>
> * __Model comparison is graded, not binary:__ Distributional discrepancy, tail-index stability, and time dependence answer different questions.
> * __Uncorrelated does not mean independent:__ Raw growth rates may have little linear memory while their magnitudes remain persistent.
> * __SIM and stress generators have different roles:__ SIM compresses cross-sectional covariance; heavy-tailed and time-varying models address tail and path risk.

___

## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. Tail estimates are sample- and cutoff-sensitive, and the white-noise ACF bands are large-sample reference bands rather than a complete multiple-testing procedure.
